In [45]:
using LowLevelFEM

In [46]:
using LinearAlgebra
using SparseArrays
using IterativeSolvers

"""
    solve_linear_system(A, B; solver=:backslash, preconditioner=nothing,
                        reltol=sqrt(eps()), maxiter=size(A, 1))

Solve the linear system `A * X = B`.

`B` may contain one or multiple right-hand sides stored columnwise.
"""
function solve_linear_system(
    A,
    B;
    solver=:backslash,
    preconditioner=nothing,
    reltol=sqrt(eps()),
    maxiter=size(A, 1)
)
    if solver === :backslash
        return A \ B
    
    elseif solver === :auto_symmetric
        try
            F = cholesky(A)
            return F \ B
        catch err
            if err isa PosDefException
                @info "Cholesky factorization failed; falling back to LU."
                F = lu(sparse(A))
                return F \ B
            end
            rethrow()
        end

    elseif solver === :lu
        A0 =
            A isa Union{Symmetric,Hermitian} ?
            sparse(A) :
            A

        F = lu(A0)
        return F \ B

    elseif solver === :cholesky
        F = cholesky(A)
        return F \ B

    elseif solver === :qr
        F = qr(A)
        return F \ B

    elseif solver === :cg
        X = similar(B)

        for j in axes(B, 2)
            if preconditioner === nothing
                X[:, j] = cg(
                    A,
                    @view(B[:, j]);
                    reltol=reltol,
                    maxiter=maxiter
                )
            else
                X[:, j] = cg(
                    A,
                    @view(B[:, j]);
                    Pl=preconditioner,
                    reltol=reltol,
                    maxiter=maxiter
                )
            end
        end

        return X

    elseif solver === :gmres
        X = similar(B)

        for j in axes(B, 2)
            if preconditioner === nothing
                X[:, j] = gmres(
                    A,
                    @view(B[:, j]);
                    reltol=reltol,
                    maxiter=maxiter
                )
            else
                X[:, j] = gmres(
                    A,
                    @view(B[:, j]);
                    Pl=preconditioner,
                    reltol=reltol,
                    maxiter=maxiter
                )
            end
        end

        return X

    else
        error("Unknown linear solver: $solver.")
    end
end

solve_linear_system (generic function with 1 method)

In [47]:
"""
    default_solver(K)

Return the default linear solver associated with the matrix wrapper.
"""
default_solver(::SystemMatrix) = :backslash
default_solver(::LowLevelFEM.SymmetricSystemMatrix) = :auto_symmetric

default_solver (generic function with 2 methods)

In [48]:
"""
    prepare_singlefield_system(K, f, support)

Prepare a single-field finite element system for solution.

Returns the reduced free-DOF matrix, right-hand side matrix, prescribed
values, and DOF index sets.
"""
function prepare_singlefield_system(
    K::SystemMatrix,
    f::Union{ScalarField,VectorField,TensorField},
    support::Vector{BoundaryCondition}
)
    K.problems === nothing ||
        error("prepare_singlefield_system: K must be a single-field SystemMatrix.")

    P = K.model

    size(K.A, 1) == size(K.A, 2) ||
        error("prepare_singlefield_system: system matrix must be square.")

    size(f.a, 1) == size(K.A, 1) ||
        error("prepare_singlefield_system: incompatible matrix and vector sizes.")

    nsteps = size(f.a, 2)

    fixed = constrainedDoFs(P, support)
    free  = freeDoFs(P, support)

    xDfield = applyBoundaryConditions(
        P,
        support;
        steps=nsteps
    )

    xD = xDfield.a

    A_ff = K.A[free, free]

    B = copy(f.a[free, :])

    if !isempty(fixed)
        B .-= K.A[free, fixed] * xD[fixed, :]
    end

    return (
        A=A_ff,
        B=B,
        xD=xD,
        free=free,
        fixed=fixed
    )
end

prepare_singlefield_system (generic function with 3 methods)

In [49]:
"""
    prepare_singlefield_system(Ks, f, support)

Prepare a symmetric single-field finite element system while preserving
the symmetric matrix representation.
"""
function prepare_singlefield_system(
    Ks::LowLevelFEM.SymmetricSystemMatrix,
    f::Union{ScalarField,VectorField,TensorField},
    support::Vector{BoundaryCondition}
)
    K = Ks.parent
    P = K.model

    K.problems === nothing ||
        error("prepare_singlefield_system: K must be a single-field SystemMatrix.")

    nsteps = size(f.a, 2)

    fixed = constrainedDoFs(P, support)
    free  = freeDoFs(P, support)

    xDfield = applyBoundaryConditions(
        P,
        support;
        steps=nsteps
    )

    xD = xDfield.a

    A = Symmetric(K.A, Ks.uplo)

    B = copy(f.a[free, :])

    if !isempty(fixed)
        B .-= A[free, fixed] * xD[fixed, :]
    end

    A_ff = Symmetric(
        K.A[free, free],
        Ks.uplo
    )

    return (
        A=A_ff,
        B=B,
        xD=xD,
        free=free,
        fixed=fixed
    )
end

prepare_singlefield_system (generic function with 3 methods)

In [50]:
function prepare_singlefield_system(
    K::SystemMatrix,
    f::Union{ScalarField,VectorField,TensorField},
    support::Vector{BoundaryCondition};
    mpc::Vector{MPC}=MPC[]
)
    return _prepare_singlefield_system(
        K,
        f,
        support;
        mpc=mpc
    )
end

prepare_singlefield_system (generic function with 3 methods)

In [51]:
function prepare_singlefield_system(
    K::LowLevelFEM.SymmetricSystemMatrix,
    f::Union{ScalarField,VectorField,TensorField},
    support::Vector{BoundaryCondition};
    mpc::Vector{MPC}=MPC[]
)
    return _prepare_singlefield_system(
        K,
        f,
        support;
        mpc=mpc
    )
end

prepare_singlefield_system (generic function with 3 methods)

In [52]:
"""
    reconstruct_singlefield_solution(f, Xfree, prepared)

Reconstruct the full finite element field from the free-DOF solution.
"""
function reconstruct_singlefield_solution(
    f::Union{ScalarField,VectorField,TensorField},
    Xfree,
    prepared
)
    u = copy(f)

    fill!(u.a, 0.0)

    u.a[prepared.free, :] .= Xfree

    if !isempty(prepared.fixed)
        u.a[prepared.fixed, :] .=
            prepared.xD[prepared.fixed, :]
    end

    return u
end

reconstruct_singlefield_solution (generic function with 1 method)

In [53]:
#=
"""
    solveField_new(K, f; support=[], solver=:auto, kwargs...)

Solve a single-field finite element system.
"""
function solveField_new(
    K::Union{SystemMatrix,LowLevelFEM.SymmetricSystemMatrix},
    f::Union{ScalarField,VectorField,TensorField};
    support::Vector{BoundaryCondition}=BoundaryCondition[],
    solver=:auto,
    preconditioner=nothing,
    reltol=sqrt(eps()),
    maxiter=size(K isa LowLevelFEM.SymmetricSystemMatrix ? K.parent.A : K.A, 1)
)
    prepared =
        prepare_singlefield_system(
            K,
            f,
            support
        )

    solver0 =
        solver === :auto ?
        default_solver(K) :
        solver

    Xfree =
        solve_linear_system(
            prepared.A,
            prepared.B;
            solver=solver0,
            preconditioner=preconditioner,
            reltol=reltol,
            maxiter=maxiter
        )

    return reconstruct_singlefield_solution(
        f,
        Xfree,
        prepared
    )
end
=#

In [54]:
function solveField_new(
    K::Union{SystemMatrix,LowLevelFEM.SymmetricSystemMatrix},
    f::Union{ScalarField,VectorField,TensorField};
    support::Vector{BoundaryCondition}=BoundaryCondition[],
    mpc::Vector{MPC}=MPC[],
    solver=:auto,
    preconditioner=nothing,
    reltol=sqrt(eps()),
    maxiter=size(K isa LowLevelFEM.SymmetricSystemMatrix ? K.parent.A : K.A, 1)
)
    prepared =
        prepare_singlefield_system(
            K,
            f,
            support;
            mpc=mpc
        )

    solver0 =
        solver === :auto ?
        default_solver(K) :
        solver

    Xfree =
        solve_linear_system(
            prepared.A,
            prepared.B;
            solver=solver0,
            preconditioner=preconditioner,
            reltol=reltol,
            maxiter=maxiter
        )

    return reconstruct_singlefield_solution(
        f,
        Xfree,
        prepared
    )
end

solveField_new (generic function with 1 method)

In [55]:
"""
    field_transformation(P)

Return the prolongation and restriction matrices associated with the field.

For a full-order field this is the active identity mapping.
For a reduced-order field the polynomial-order reduction is used.
"""
function field_transformation(P::Problem)
    if P.reducedOrder
        return reductionMatrices(P)
    else
        return LowLevelFEM.active_identity(P)
    end
end

field_transformation (generic function with 2 methods)

In [56]:
"""
    field_transformation(P, mpc=MPC[])

Return the complete field transformation including polynomial-order
reduction and multi-point constraints.
"""
function field_transformation(
    P::Problem,
    mpc::Vector{MPC}=MPC[]
)
    return LowLevelFEM._singlefield_mpc_transformation(
        P,
        mpc
    )
end

field_transformation (generic function with 2 methods)

In [57]:
"""
    singlefield_mpc_bc_data_matrix(P, mpc, fixed, xD)

Transfer prescribed values from MPC slave DOFs to their final master DOFs
for one or multiple right-hand sides.
"""
function singlefield_mpc_bc_data_matrix(
    P::Problem,
    mpc::Vector{MPC},
    fixed::Vector{Int},
    xD::AbstractMatrix
)
    isempty(mpc) && return fixed, xD

    nsteps = size(xD, 2)

    xD_eff =
        zeros(
            eltype(xD),
            size(xD)
        )

    fixed_eff = Int[]

    for j in 1:nsteps

        fixed_j, xD_j =
            LowLevelFEM._singlefield_mpc_bc_data(
                P,
                mpc,
                fixed,
                @view xD[:, j]
            )

        if j == 1
            fixed_eff = fixed_j
        elseif fixed_j != fixed_eff
            error(
                "MPC: inconsistent constrained DOFs between right-hand sides."
            )
        end

        xD_eff[:, j] .= xD_j
    end

    return fixed_eff, xD_eff
end

singlefield_mpc_bc_data_matrix (generic function with 1 method)

In [58]:
"""
    reduced_bc_data_matrix(R, fixed, xD)

Map full-space Dirichlet data with one or multiple right-hand sides
to the reduced space.
"""
function reduced_bc_data_matrix(
    R::SparseMatrixCSC,
    fixed::AbstractVector{<:Integer},
    xD::AbstractMatrix
)
    nr = size(R, 1)

    if isempty(fixed)
        return (
            collect(1:nr),
            Int[],
            zeros(eltype(xD), nr, size(xD, 2))
        )
    end

    # The constrained reduced DOFs depend only on the set of
    # constrained full-space DOFs, not on their prescribed values.
    free_r, fixed_r, _ =
        reduced_bc_data(
            R,
            fixed,
            @view xD[:, 1]
        )

    xD_r = R * xD

    return free_r, fixed_r, xD_r
end

reduced_bc_data_matrix (generic function with 1 method)

In [59]:
parent_system(K::SystemMatrix) = K
parent_system(K::LowLevelFEM.SymmetricSystemMatrix) = K.parent

reduced_matrix_wrapper(::SystemMatrix, A) = A

reduced_matrix_wrapper(
    K::LowLevelFEM.SymmetricSystemMatrix,
    A
) = Symmetric(A, K.uplo)

reduced_matrix_wrapper (generic function with 2 methods)

In [60]:
"""
    prepare_singlefield_system(K, f, support)

Prepare a single-field finite element system for solution.

The same transformation pipeline is used for full-order and reduced-order
fields.
"""
function prepare_singlefield_system(
    K0::Union{SystemMatrix,LowLevelFEM.SymmetricSystemMatrix},
    f::Union{ScalarField,VectorField,TensorField},
    support::Vector{BoundaryCondition}
)
    K = parent_system(K0)
    P = K.model

    K.problems === nothing ||
        error(
            "prepare_singlefield_system: K must be a single-field SystemMatrix."
        )

    size(K.A, 1) == size(K.A, 2) ||
        error(
            "prepare_singlefield_system: system matrix must be square."
        )

    size(f.a, 1) == size(K.A, 1) ||
        error(
            "prepare_singlefield_system: incompatible matrix and vector sizes."
        )

    nsteps = size(f.a, 2)

    # ----------------------------------------------------------
    # 1) Full-space Dirichlet data
    # ----------------------------------------------------------

    fixed = constrainedDoFs(P, support)

    xDfield =
        applyBoundaryConditions(
            P,
            support;
            steps=nsteps
        )

    xD = xDfield.a

    # ----------------------------------------------------------
    # 2) Field transformation
    # ----------------------------------------------------------

    T, R = field_transformation(P)

    # ----------------------------------------------------------
    # 3) Dirichlet data in transformed space
    # ----------------------------------------------------------

    free_r, fixed_r, xD_r =
        reduced_bc_data_matrix(
            R,
            fixed,
            xD
        )

    # ----------------------------------------------------------
    # 4) Galerkin projection
    # ----------------------------------------------------------

    Kr = T' * K.A * T
    Br = T' * f.a

    # ----------------------------------------------------------
    # 5) Apply prescribed values
    # ----------------------------------------------------------

    B = copy(Br[free_r, :])

    if !isempty(fixed_r)
        B .-=
            Kr[free_r, fixed_r] *
            xD_r[fixed_r, :]
    end

    A =
        reduced_matrix_wrapper(
            K0,
            Kr[free_r, free_r]
        )

    return (
        A=A,
        B=B,
        T=T,
        xD_r=xD_r,
        free_r=free_r,
        fixed_r=fixed_r,
        nr=size(T, 2)
    )
end

prepare_singlefield_system (generic function with 3 methods)

In [61]:
"""
    reconstruct_singlefield_solution(f, Xfree, prepared)

Reconstruct the full finite element field from the transformed solution.
"""
function reconstruct_singlefield_solution(
    f::Union{ScalarField,VectorField,TensorField},
    Xfree,
    prepared
)
    nsteps = size(Xfree, 2)

    xr =
        zeros(
            eltype(Xfree),
            prepared.nr,
            nsteps
        )

    xr[prepared.free_r, :] .= Xfree

    if !isempty(prepared.fixed_r)
        xr[prepared.fixed_r, :] .=
            prepared.xD_r[prepared.fixed_r, :]
    end

    x = prepared.T * xr

    u = copy(f)
    u.a .= x

    return u
end

reconstruct_singlefield_solution (generic function with 1 method)

In [62]:
#=
function solveField_new(
    K::Union{SystemMatrix,SymmetricSystemMatrix},
    f::Union{ScalarField,VectorField,TensorField};
    support::Vector{BoundaryCondition}=BoundaryCondition[],
    solver=:auto,
    preconditioner=nothing,
    reltol=sqrt(eps()),
    maxiter=size(K isa SymmetricSystemMatrix ? K.parent.A : K.A, 1)
)
    prepared =
        prepare_singlefield_system(
            K,
            f,
            support
        )

    solver0 =
        solver === :auto ?
        default_solver(K) :
        solver

    Xfree =
        solve_linear_system(
            prepared.A,
            prepared.B;
            solver=solver0,
            preconditioner=preconditioner,
            reltol=reltol,
            maxiter=maxiter
        )

    return reconstruct_singlefield_solution(
        f,
        Xfree,
        prepared
    )
end
=#

In [63]:
#=
function _prepare_singlefield_system(
    K0,
    f,
    support
)
    K = parent_system(K0)
    P = K.model

    K.problems === nothing ||
        error(
            "prepare_singlefield_system: K must be a single-field SystemMatrix."
        )

    size(K.A, 1) == size(K.A, 2) ||
        error(
            "prepare_singlefield_system: system matrix must be square."
        )

    size(f.a, 1) == size(K.A, 1) ||
        error(
            "prepare_singlefield_system: incompatible matrix and vector sizes."
        )

    nsteps = size(f.a, 2)

    fixed = constrainedDoFs(P, support)

    xDfield =
        applyBoundaryConditions(
            P,
            support;
            steps=nsteps
        )

    xD = xDfield.a

    T, R = field_transformation(P)

    free_r, fixed_r, xD_r =
        reduced_bc_data_matrix(
            R,
            fixed,
            xD
        )

    Kr = T' * K.A * T
    Br = T' * f.a

    B = copy(Br[free_r, :])

    if !isempty(fixed_r)
        B .-=
            Kr[free_r, fixed_r] *
            xD_r[fixed_r, :]
    end

    A =
        reduced_matrix_wrapper(
            K0,
            Kr[free_r, free_r]
        )

    return (
        A=A,
        B=B,
        T=T,
        xD_r=xD_r,
        free_r=free_r,
        fixed_r=fixed_r,
        nr=size(T, 2)
    )
end
=#

In [64]:
function _prepare_singlefield_system(
    K0,
    f,
    support;
    mpc::Vector{MPC}=MPC[]
)
    K = parent_system(K0)
    P = K.model

    K.problems === nothing ||
        error(
            "prepare_singlefield_system: K must be a single-field SystemMatrix."
        )

    size(K.A, 1) == size(K.A, 2) ||
        error(
            "prepare_singlefield_system: system matrix must be square."
        )

    size(f.a, 1) == size(K.A, 1) ||
        error(
            "prepare_singlefield_system: incompatible matrix and vector sizes."
        )

    nsteps = size(f.a, 2)

    # ----------------------------------------------------------
    # 1) Full-space Dirichlet data
    # ----------------------------------------------------------

    fixed = constrainedDoFs(P, support)

    xDfield =
        applyBoundaryConditions(
            P,
            support;
            steps=nsteps
        )

    xD = xDfield.a

    # ----------------------------------------------------------
    # 2) Complete field transformation
    #    full order / P1-P2 / MPC / combinations
    # ----------------------------------------------------------

    T, R =
        field_transformation(
            P,
            mpc
        )

    # ----------------------------------------------------------
    # 3) Transfer BC data through MPC relations
    # ----------------------------------------------------------

    fixed_eff, xD_eff =
        singlefield_mpc_bc_data_matrix(
            P,
            mpc,
            fixed,
            xD
        )

    # ----------------------------------------------------------
    # 4) Dirichlet data in transformed space
    # ----------------------------------------------------------

    free_r, fixed_r, xD_r =
        reduced_bc_data_matrix(
            R,
            fixed_eff,
            xD_eff
        )

    # ----------------------------------------------------------
    # 5) Galerkin projection
    # ----------------------------------------------------------

    Kr = T' * K.A * T
    Br = T' * f.a

    # ----------------------------------------------------------
    # 6) Apply prescribed values
    # ----------------------------------------------------------

    B = copy(Br[free_r, :])

    if !isempty(fixed_r)
        B .-=
            Kr[free_r, fixed_r] *
            xD_r[fixed_r, :]
    end

    A =
        reduced_matrix_wrapper(
            K0,
            Kr[free_r, free_r]
        )

    return (
        A=A,
        B=B,
        T=T,
        xD_r=xD_r,
        free_r=free_r,
        fixed_r=fixed_r,
        nr=size(T, 2)
    )
end

_prepare_singlefield_system (generic function with 1 method)

In [65]:
prepare_singlefield_system(
    K::SystemMatrix,
    f::Union{ScalarField,VectorField,TensorField},
    support::Vector{BoundaryCondition}
) =
    _prepare_singlefield_system(K, f, support)

prepare_singlefield_system (generic function with 3 methods)

In [66]:
prepare_singlefield_system(
    K::LowLevelFEM.SymmetricSystemMatrix,
    f::Union{ScalarField,VectorField,TensorField},
    support::Vector{BoundaryCondition}
) =
    _prepare_singlefield_system(K, f, support)

prepare_singlefield_system (generic function with 3 methods)

In [67]:
structured_box_mesh()

mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u)

K = ∫(ε(U) ⋅ D(:Solid, mat) ⋅ ε(U))
f = ∫(U ⋅ [1,0,0], Γ="right")
support = [BoundaryCondition("left", ux=0.01, uy=0, uz=0)]

1-element Vector{BoundaryCondition}:
 BoundaryCondition("left", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0, :uz => 0, :ux => 0.01))

In [68]:
u_old = solveField(
    K,
    f;
    support=support
)

u_new = solveField_new(
    K,
    f;
    support=support
)

norm(u_old.a - u_new.a)

6.712186201863951e-16

In [69]:
us_old = solveField(
    Symmetric(K),
    f;
    support=support
)

us_new = solveField_new(
    Symmetric(K),
    f;
    support=support
)

norm(us_old.a - us_new.a)

2.8803466837580606e-16

In [70]:
u_lu = solveField_new(
    Symmetric(K),
    f;
    support=support,
    solver=:lu
)

u_chol = solveField_new(
    Symmetric(K),
    f;
    support=support,
    solver=:cholesky
)

(
    norm(u_new.a - u_lu.a),
    norm(u_new.a - u_chol.a)
)

(7.699625828096422e-16, 7.253952967820608e-16)

In [71]:
u_old = solveField(
    K,
    f;
    support=support
)

u_new = solveField_new(
    K,
    f;
    support=support
)

norm(u_old.a - u_new.a)

6.712186201863951e-16

In [72]:
us_old = solveField(
    Symmetric(K),
    f;
    support=support
)

us_new = solveField_new(
    Symmetric(K),
    f;
    support=support
)

norm(us_old.a - us_new.a)

2.8803466837580606e-16

In [73]:
structured_box_mesh(order=2)

mat = Material("body")
Ur = Field([mat], type=:VectorField, dim=3, fieldName=:u, reducedOrder=true)

Kr = ∫(ε(Ur) ⋅ D(:Solid, mat) ⋅ ε(Ur))
fr = ∫(Ur ⋅ [1,0,0], Γ="right")
support = [BoundaryCondition("left", ux=0.01, uy=0, uz=0)]

1-element Vector{BoundaryCondition}:
 BoundaryCondition("left", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0, :uz => 0, :ux => 0.01))

In [74]:
u_old = solveField(
    Kr,
    fr;
    support=support
)

u_new = solveField_new(
    Kr,
    fr;
    support=support
)

(
    size(u_old.a),
    size(u_new.a),
    norm(u_old.a - u_new.a)
)

((27783, 1), (27783, 1), 1.864866056738379e-15)

In [75]:
u_new_sym = solveField_new(
    Symmetric(Kr),
    fr;
    support=support
)

norm(u_old.a - u_new_sym.a)

2.5340501185153356e-15

In [76]:
T, R = reductionMatrices(Ur)

(
    size(Kr.A),
    size(T),
    size(R)
)

((27783, 27783), (27783, 3993), (3993, 27783))

In [77]:
prepared = prepare_singlefield_system(
    Kr,
    fr,
    support
)

(
    size(Kr.A),
    size(prepared.T),
    size(prepared.A)
)

((27783, 27783), (27783, 3993), (3630, 3630))

In [78]:
openGeometry("mpc-1.geo")

#openPreProcessor()
mat1 = Material("body")
mat2 = Material("remote")

U = Field([mat1, mat2], type=:VectorField, dim=2, fieldName=:u, rhsName=:f)
Ku = ∫(SymGrad(U) ⋅ [2 1 0; 1 2 0; 0 0 1] ⋅ SymGrad(U), Ω="body")
mpc = MPC(master="remote", slave="right", uy=false)
bc2 = BoundaryCondition("remote", uy=0)
bc = BoundaryCondition("left", ux=0, uy=0)

fu = ∫(U ⋅ [1, 0], Γ="remote")
u_old = solveField(
    Ku,
    fu;
    support=[bc, bc2],
    mpc=[mpc]
)

u_new = solveField_new(
    Ku,
    fu;
    support=[bc, bc2],
    mpc=[mpc]
)

(
    norm(u_old.a - u_new.a),
    norm(u_old.a - u_new.a) / norm(u_old.a)
)

(0.0, 0.0)

In [79]:
structured_box_mesh()

mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u)

K = ∫(ε(U) ⋅ D(:Solid, mat) ⋅ ε(U))
f = ∫(U ⋅ [1,0,0], Γ="right")
support = [BoundaryCondition("left", ux=0.01, uy=0, uz=0)]
u_old = solveField(
    K,
    f;
    support=support
)

u_new = solveField_new(
    K,
    f;
    support=support
)

norm(u_old.a - u_new.a)
us_old = solveField(
    Symmetric(K),
    f;
    support=support
)

us_new = solveField_new(
    Symmetric(K),
    f;
    support=support
)

norm(us_old.a - us_new.a)
u_lu = solveField_new(
    Symmetric(K),
    f;
    support=support,
    solver=:lu
)

u_chol = solveField_new(
    Symmetric(K),
    f;
    support=support,
    solver=:cholesky
)

(
    norm(u_new.a - u_lu.a),
    norm(u_new.a - u_chol.a)
)

(7.699625828096422e-16, 7.253952967820608e-16)

In [80]:
structured_box_mesh()
Premote = gmsh.model.occ.addPoint(1, 0.5, 0.5)
gmsh.model.occ.synchronize()
gmsh.model.addPhysicalGroup(0, [Premote], -1, "remote")

gmsh.model.mesh.clear()
gmsh.model.mesh.generate(3)

In [81]:
mat = Material("body")
rem = Material("remote")
U = Field([mat,rem], type=:VectorField, dim=3, fieldName=:u)
Ku = ∫(ε(U) ⋅ D(:Solid, mat) ⋅ ε(U), Ω="body")
fu = ∫(U ⋅ [1,0,0], Γ="right")

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [87]:
u_sym = solveField_new(
    Symmetric(Ku),
    fu;
    support=[bc, bc2],
    mpc=[mpc]
)

nonrem = constrainedDoFs(U, [BoundaryCondition("body", ux=0, uy=0, uz=0)])
norm(u_old.a - u_sym.a[nonrem])

0.36487725951730426